# NLP Practical Exam — Text Processing + Language Modeling (90 minutes)

**Instructions**
- Work in this notebook only.
- Write short, clear comments to justify *tool choices* (regex vs NLTK, etc.).
- Do **not** use external NLP libraries beyond **NLTK**, **NumPy**, **PyTorch** (PyTorch not needed here).
- Keep outputs readable (print key variables).

**Total: 10 points**


## Given text

```python
text = ("In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. He is 1.86m tall and met with researchers from U.P.C. and U.N.E.S.C.O. A report valued the project at $3.2 billion.")
```

> Treat the text as *synthetic exam data* (no fact-checking needed).


## Questions

1. **(1 pt)** Sentence splitting using **regex + NLTK**.
2. **(1 pt)** Regex normalization: acronyms, height meters→centimeters, money `$X.Y billion` → `x point y billion` (words).
3. **(1 pt)** Lowercase **except** proper nouns; join multiword proper nouns with underscore (e.g., `Sam Altman → Sam_Altman`). Keep acronyms uppercase.
4. **(1 pt)** Tokenize (tool of your choice).
5. **(1 pt)** Remove stopwords (tool of your choice); keep entity tokens.
6. **(1 pt)** Create bigrams with pure Python.
7. **(2 pt)** Build a bigram LM (MLE) and `predict_next(prev_word, top_k=3)`.

8. **(2 pt)** Implement a simple **BPE** on: `corpus = "low lower newest widest"` (≥5 merges or until no merges).
9. **(1 pt)** Compute Accuracy/Precision/Recall/F1 for an invented confusion matrix (explain with comments).


In [1]:
import re
import math
import nltk
from collections import Counter, defaultdict

# NLTK downloads (safe to run multiple times)
nltk.download("punkt", quiet=True)
nltk.download("stopwords", quiet=True)

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

text = ("In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. "
        "He is 1.86m tall and met with researchers from U.P.C. and U.N.E.S.C.O. "
        "A report valued the project at $3.2 billion.")

print(text)


In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. He is 1.86m tall and met with researchers from U.P.C. and U.N.E.S.C.O. A report valued the project at $3.2 billion.


## Q1

In [12]:
# Q1 (1 pt): Sentence splitting (regex + NLTK)
# - Use regex to protect acronyms like U.P.C. so they don't break sentence boundaries.
# - Then use nltk.sent_tokenize.
#
# Return: sentences (list of strings)

# TODO: implement protect_acronym_dots and restore_acronym_dots (or equivalent)
# TODO: apply sent_tokenize

def protect_acronym_dots(text):
    """
    Protect dots in acronyms (e.g., U.P.C.) by replacing them with a placeholder.
    """
    return re.sub(r'\b([A-Z]\.){2,}(?!\s+[A-Z])', lambda m: m.group().replace('.', '<dot>'), text)

def restore_acronym_dots(text):
    """
    Restore the original dots in acronyms by replacing the placeholder back to a dot.
    """
    return text.replace('<dot>', '.')

sentences = sent_tokenize(protect_acronym_dots(text)) # Splitting the sentences after protecting the acronym dots
sentences = [restore_acronym_dots(sentence) for sentence in sentences] # Restoring the original dots in the sentences

print(sentences)


['In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona.', 'He is 1.86m tall and met with researchers from U.P.C. and U.N.E.S.C.O.', 'A report valued the project at $3.2 billion.']


## Q2

In [14]:
# Q2 (1 pt): Regex normalization
# Convert:
#  - U.P.C. -> UPC, U.N.E.S.C.O. -> UNESCO (general rule: remove dots in acronyms)
#  - 1.86m -> 186 centimeters (general: X.YZm -> int(round(float(X.YZ)*100)) centimeters)
#  - $3.2 billion -> three point two billion  (digits 0-9 are enough)
#
# Return: text_norm

def normalize_text(text):
    """
    Normalize the text by applying the specified transformations:
    """
    
    def money_replacer(match):
        """
        Convert money to words.
        """
        digit_map = {'0':'zero', '1':'one', '2':'two', '3':'three', '4':'four',
                     '5':'five', '6':'six', '7':'seven', '8':'eight', '9':'nine'}
        integer_part = match.group(1) # Splitting the integer and decimal parts
        decimal_part = match.group(2)
        return f"{digit_map[integer_part]} point {digit_map[decimal_part]} billion"
    
    def meters_replacer(match):
        """
        Convert meters to centimeters.
        """
        meters = float(match.group(1))
        cm = int(round(meters * 100))
        return f"{cm} centimeters"

    text = re.sub(r'\b([A-Z]\.){2,}(?!\s+[A-Z])', lambda m: m.group().replace('.', ''), text) # Removing dots in acronyms

    text = re.sub(r'(\d+\.\d+)m\b', meters_replacer, text) # Converting meters to centimeters
    
    text = re.sub(r'\$(\d)\.(\d) billion', money_replacer, text) # Converting money to words
    
    return text

text_norm = normalize_text(text)

print(text_norm)


In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. He is 186 centimeters tall and met with researchers from UPC and UNESCO. A report valued the project at three point two billion.


## Q3

In [17]:
# Q3 (1 pt): Lowercase except proper nouns + underscore multiword proper nouns
# Requirements:
# - Convert to lowercase except:
#   - Acronyms (ALL CAPS) stay uppercase (e.g., UNESCO, UPC, CEO)
#   - MixedCase tokens stay as-is (e.g., OpenAI)
#   - Multiword proper nouns joined with underscore (Sam Altman -> Sam_Altman) and preserved
#
# Return: text_case

def lowercasing_text(text):
    """
    Lowercase the text while preserving acronyms, mixed case tokens, and multiword proper nouns with underscores.
    """
    text = re.sub(r'\b([A-Z][a-z]+)\s+([A-Z][a-z]+)\b', r'\1_\2', text)
    
    tokens = text.split()
    result = []
    
    for token in tokens:
        word = re.match(r'([^\s,\.!?]+)([\.,!?]*)', token)
        if word:
            w = word.group(1)
            punct = word.group(2)
            
            if w.isupper() and len(w) > 1:
                result.append(w + punct)
            elif any(c.isupper() for c in w[1:]):
                result.append(w + punct)
            elif '_' in w:
                result.append(w + punct)
            elif w[0].isupper() and len(w) > 1:
                result.append(w + punct)
            else:
                result.append(w.lower() + punct)
        else:
            result.append(token)
    
    return ' '.join(result)

text_case = lowercasing_text(text_norm)
print(text_case)



in mid-February 2026, the CEO of OpenAI, Sam_Altman, visited Barcelona. he is 186 centimeters tall and met with researchers from UPC and UNESCO. a report valued the project at three point two billion.


## Q4

In [23]:
# Q4 (1 pt): Tokenization
# Use a tokenizer of your choice (e.g., nltk.word_tokenize).
# Return: tokens (list)

nltk.download("punkt_tab")

def tokenize_text(text):
    """
    Tokenize the text using nltk's word_tokenize.
    """
    return word_tokenize(text) # NLTK's word tokenizer

tokens = tokenize_text(text_case)

print(tokens)


['in', 'mid-February', '2026', ',', 'the', 'CEO', 'of', 'OpenAI', ',', 'Sam_Altman', ',', 'visited', 'Barcelona', '.', 'he', 'is', '186', 'centimeters', 'tall', 'and', 'met', 'with', 'researchers', 'from', 'UPC', 'and', 'UNESCO', '.', 'a', 'report', 'valued', 'the', 'project', 'at', 'three', 'point', 'two', 'billion', '.']


[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/mariotg/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## Q5

In [27]:
# Q5 (1 pt): Stopword removal
# - Remove English stopwords
# - Do NOT remove entity tokens like OpenAI, Sam_Altman, Barcelona, UNESCO, UPC
# Return: tokens_nostop
from nltk.corpus import stopwords

def remove_stopwords(tokens):
    """
    Removing stopwords from a given text
    """
    stop_words = set(stopwords.words('english')) # Getting the set of English stopwords
    
    tokens_nostop = [] # List to hold tokens that are not stopwords
    
    for token in tokens:
        if token.lower() not in stop_words or token[0].isupper() or '_' in token or token.isdigit(): # Checking if the token is not a stopword or is an entity token
            tokens_nostop.append(token)
    
    return tokens_nostop
    

tokens_nostop = remove_stopwords(tokens)

print(tokens_nostop)


['mid-February', '2026', ',', 'CEO', 'OpenAI', ',', 'Sam_Altman', ',', 'visited', 'Barcelona', '.', '186', 'centimeters', 'tall', 'met', 'researchers', 'UPC', 'UNESCO', '.', 'report', 'valued', 'project', 'three', 'point', 'two', 'billion', '.']


## Q6

In [25]:
# Q6 (1 pt): Bigrams with pure Python (no NLTK bigrams helper)
# Return: bigrams = [(w1, w2), ...]

def make_bigrams(text):
    """
    Create bigrams from the given text.
    """
    tokens = word_tokenize(text) # Tokenizing the text
    
    bigrams = []
    
    for i in range(len(tokens) - 1):
        bigrams.append((tokens[i], tokens[i + 1])) # Creating bigrams by pairing each token with the next one

    return bigrams

bigrams = make_bigrams(text)

print(bigrams)


[('In', 'mid-February'), ('mid-February', '2026'), ('2026', ','), (',', 'the'), ('the', 'CEO'), ('CEO', 'of'), ('of', 'OpenAI'), ('OpenAI', ','), (',', 'Sam'), ('Sam', 'Altman'), ('Altman', ','), (',', 'visited'), ('visited', 'Barcelona'), ('Barcelona', '.'), ('.', 'He'), ('He', 'is'), ('is', '1.86m'), ('1.86m', 'tall'), ('tall', 'and'), ('and', 'met'), ('met', 'with'), ('with', 'researchers'), ('researchers', 'from'), ('from', 'U.P.C'), ('U.P.C', '.'), ('.', 'and'), ('and', 'U.N.E.S.C.O'), ('U.N.E.S.C.O', '.'), ('.', 'A'), ('A', 'report'), ('report', 'valued'), ('valued', 'the'), ('the', 'project'), ('project', 'at'), ('at', '$'), ('$', '3.2'), ('3.2', 'billion'), ('billion', '.')]


## Q7

In [ ]:
# Q7 (2 pt): Bigram Language Model + next-word prediction
# Build:
# - bigram_counts[(w1,w2)]
# - context_counts[w1]
# - model[w1][w2] = P(w2|w1) = count(w1,w2)/count(w1)
#
# Then implement:
# def predict_next(prev_word, model, top_k=3): -> list[(next_word, prob)] sorted

bigram_counts = None
context_counts = None
model = None

def predict_next(prev_word, model, top_k=3):
    # TODO
    return None

# Example:
# print(predict_next("OpenAI", model, top_k=3))


## Q8

In [ ]:
# Q8 (2 pt): Simple BPE (Byte Pair Encoding) on a tiny corpus
corpus = "low lower newest widest"

# Requirements:
# - Represent each word as characters + </w>
# - Compute pair frequencies (weighted by word frequency)
# - Merge most frequent pair
# - Do at least 5 merges (or stop if no pairs)
#
# Deliver:
# - merges: list of merges in order
# - final segmented version of each word

merges = None

# TODO: implement BPE helper functions:
# - get_vocab_from_corpus
# - get_pair_frequencies
# - merge_pair_in_vocab

# print(merges)


## Q9

In [ ]:
# Q9 (1 pt): Metrics — Accuracy, Precision, Recall, F1
# Invent a confusion matrix (TP, FP, FN, TN) and compute metrics.
# Explain each formula briefly in comments.

TP = None
FP = None
FN = None
TN = None

accuracy = None
precision = None
recall = None
f1 = None

# print(accuracy, precision, recall, f1)
